In [ ]:
from google.colab import drive

# Montar Google Drive en Colab
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from google.colab import drive

# 1. Asegurar que Google Drive esté montado y que la carpeta exista
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/practica2"
os.makedirs(BASE_DIR, exist_ok=True)

ZIP_PATH = os.path.join(BASE_DIR, "Animals-10.zip")
EXTRACT_DIR = "/content/animals10"

# 2. Descomprimir el dataset si no está en la sesión
if not os.path.exists(EXTRACT_DIR):
    print("Descomprimiendo el dataset...")
    !unzip -q "{ZIP_PATH}" -d "{EXTRACT_DIR}"
    print("Dataset descomprimido con éxito.")

# 3. Parámetros del modelo
SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 32
NUM_CLASSES = 10

# 4. Cargar las rutas de train y val guardadas por el Integrante 1
train_df = pd.read_csv(f"{BASE_DIR}/train.csv")
val_df = pd.read_csv(f"{BASE_DIR}/val.csv")

# 5. Función de lectura y normalización de imágenes
def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, tf.one_hot(etiqueta, NUM_CLASSES)

# 6. Data Augmentation
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def crear_dataset(df, es_entrenamiento=False):
    ds = tf.data.Dataset.from_tensor_slices((df["ruta"].values, df["etiqueta"].values))
    ds = ds.map(cargar_imagen, num_parallel_calls=tf.data.AUTOTUNE)
    if es_entrenamiento:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = crear_dataset(train_df, es_entrenamiento=True)
val_ds = crear_dataset(val_df, es_entrenamiento=False)

# 7. Arquitectura de la CNN (Corregida para Keras 3)
modelo_cnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name="CNN_Animales_10")

modelo_cnn.summary()

# 8. Compilación y Callbacks guardando en disco local (/content/)
LOCAL_MODEL_PATH = "/content/modelo_cnn_animales.keras"

modelo_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_list = [
    callbacks.ModelCheckpoint(
        filepath=LOCAL_MODEL_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]

# 9. Entrenamiento
print("\nIniciando entrenamiento del modelo...")
historia = modelo_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_list
)

# 10. Guardar historial localmente y copiar todo a Google Drive
LOCAL_HIST_PATH = "/content/historial_entrenamiento.csv"
hist_df = pd.DataFrame(historia.history)
hist_df.to_csv(LOCAL_HIST_PATH, index=False)

print("\nCopiando artefactos finales a Google Drive...")
shutil.copy(LOCAL_MODEL_PATH, f"{BASE_DIR}/modelo_cnn_animales.keras")
shutil.copy(LOCAL_HIST_PATH, f"{BASE_DIR}/historial_entrenamiento.csv")

print("\n========================================================")
print("¡ ENTRENAMIENTO FINALIZADO Y ARCHIVOS GUARDADOS EN DRIVE !")
print(f"1. Modelo: {BASE_DIR}/modelo_cnn_animales.keras")
print(f"2. Historial: {BASE_DIR}/historial_entrenamiento.csv")
print("========================================================")

Mounted at /content/drive


Model: "CNN_Animales_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,423,306 (5.43 MB)

 Trainable params: 1,423,306 (5.43 MB)

 Non-trainable params: 0 (0.00 B)


Iniciando entrenamiento del modelo...
Epoch 1/20
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.2355 - loss: 2.1338
Epoch 1: val_loss improved from None to 1.78357, saving model to /content/modelo_cnn_animales.keras

Epoch 1: finished saving model to /content/modelo_cnn_animales.keras
655/655 ━━━━━━━━━━━━━━━━━━━━ 115s 168ms/step - accuracy: 0.2878 - loss: 2.0098 - val_accuracy: 0.4121 - val_loss: 1.7836 - learning_rate: 0.0010
Epoch 2/20
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.4131 - loss: 1.7050
Epoch 2: val_loss improved from 1.78357 to 1.54659, saving model to /content/modelo_cnn_animales.keras

Epoch 2: finished saving model to /content/modelo_cnn_animales.keras
655/655 ━━━━━━━━━━━━━━━━━━━━ 102s 155ms/step - accuracy: 0.4430 - loss: 1.6301 - val_accuracy: 0.4691 - val_loss: 1.5466 - learning_rate: 0.0010
Epoch 3/20
655/655 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.5035 - loss: 1.4613
Epoch 3: val_loss improved from 1.54659 to 1.48064, saving model

OSError: [Errno 30] Read-only file system: '/content/drive/MyDrive/practica2/modelo_cnn_animales.keras'

In [ ]:
import os
import shutil
from google.colab import files

# 1. Crear una carpeta PROPIA
MI_CARPETA_PROPIA = "/content/drive/MyDrive/practica2_mi_unidad"
os.makedirs(MI_CARPETA_PROPIA, exist_ok=True)

# 2. Copiar el modelo y el historial a tu propio Drive
shutil.copy("/content/modelo_cnn_animales.keras", f"{MI_CARPETA_PROPIA}/modelo_cnn_animales.keras")
shutil.copy("/content/historial_entrenamiento.csv", f"{MI_CARPETA_PROPIA}/historial_entrenamiento.csv")

print(f"✅ ¡Archivos guardados con éxito en tu Google Drive!: {MI_CARPETA_PROPIA}")

# 3. Descargar copias directamente a tu computadora (opcional)
print("Descargando archivos a tu computadora...")
files.download("/content/modelo_cnn_animales.keras")
files.download("/content/historial_entrenamiento.csv")

✅ ¡Archivos guardados con éxito en tu Google Drive!: /content/drive/MyDrive/practica2_mi_unidad
Descargando archivos a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>